In [1]:
# 1. Cài đặt pyspark
!pip install pyspark

In [5]:
# 2. Khởi tạo SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Khởi tạo Spark Session
spark = SparkSession.builder.appName("PySparkBasics").getOrCreate()

In [6]:
# 1. Tạo dữ liệu giả lập (Thay cho bước đọc file)
data = [("Dien thoai", "Dien tu", 1000, 2),
        ("Tu lanh", "Gia dung", 1500, 1),
        ("Laptop", "Dien tu", 2000, 5),
        ("May giat", "Gia dung", 1200, 2),
        ("Chuot", "Dien tu", 50, 10)]

columns = ["TenSanpham", "DanhMuc", "Gia", "SoLuong"]

# Tạo DataFrame
df = spark.createDataFrame(data, columns)

# 2. Hiển thị dữ liệu (Giống df.head() trong Pandas)
print("Hiển thị 5 dòng đầu tiên:")
df.show(5)

Hiển thị 5 dòng đầu tiên:
+----------+--------+----+-------+
|TenSanpham| DanhMuc| Gia|SoLuong|
+----------+--------+----+-------+
|Dien thoai| Dien tu|1000|      2|
|   Tu lanh|Gia dung|1500|      1|
|    Laptop| Dien tu|2000|      5|
|  May giat|Gia dung|1200|      2|
|     Chuot| Dien tu|  50|     10|
+----------+--------+----+-------+



In [7]:
# 3. Xem cấu trúc dữ liệu (Giống df.info())
print("Cấu trúc schema:")
df.printSchema()

Cấu trúc schema:
root
 |-- TenSanpham: string (nullable = true)
 |-- DanhMuc: string (nullable = true)
 |-- Gia: long (nullable = true)
 |-- SoLuong: long (nullable = true)



In [8]:
# 4. Thống kê mô tả (Giống df.describe())
print("Thống kê mô tả:")
df.describe(["Gia", "SoLuong"]).show()

Thống kê mô tả:
+-------+-----------------+------------------+
|summary|              Gia|           SoLuong|
+-------+-----------------+------------------+
|  count|                5|                 5|
|   mean|           1150.0|               4.0|
| stddev|721.1102550927978|3.6742346141747673|
|    min|               50|                 1|
|    max|             2000|                10|
+-------+-----------------+------------------+



In [10]:
# 5. Thao tác chọn cột và tạo cột mới (Giống df['new_col'] = ...)
# Tính tổng tiền = Gia * SoLuong
df_with_total = df.withColumn("TongTien", F.col("Gia") * F.col("SoLuong"))
df_with_total.show()

+----------+--------+----+-------+--------+
|TenSanpham| DanhMuc| Gia|SoLuong|TongTien|
+----------+--------+----+-------+--------+
|Dien thoai| Dien tu|1000|      2|    2000|
|   Tu lanh|Gia dung|1500|      1|    1500|
|    Laptop| Dien tu|2000|      5|   10000|
|  May giat|Gia dung|1200|      2|    2400|
|     Chuot| Dien tu|  50|     10|     500|
+----------+--------+----+-------+--------+



In [11]:
# 6. Lọc dữ liệu (Giống df[df['Gia'] > 1000])
print("Các sản phẩm có giá trên 1000:")
df_with_total.filter(F.col("Gia") > 1000).show()

Các sản phẩm có giá trên 1000:
+----------+--------+----+-------+--------+
|TenSanpham| DanhMuc| Gia|SoLuong|TongTien|
+----------+--------+----+-------+--------+
|   Tu lanh|Gia dung|1500|      1|    1500|
|    Laptop| Dien tu|2000|      5|   10000|
|  May giat|Gia dung|1200|      2|    2400|
+----------+--------+----+-------+--------+



In [12]:
# 7. Gom nhóm và Tính toán (Aggregation - Giống df.groupby().agg())
print("Tổng doanh thu theo từng Danh mục:")
summary_df = df_with_total.groupBy("DanhMuc").agg(
    F.sum("TongTien").alias("TongDoanhThu"),
    F.avg("Gia").alias("GiaTrungBinh"),
    F.count("TenSanpham").alias("SoLuongMatHang")
)
summary_df.show()

Tổng doanh thu theo từng Danh mục:
+--------+------------+------------------+--------------+
| DanhMuc|TongDoanhThu|      GiaTrungBinh|SoLuongMatHang|
+--------+------------+------------------+--------------+
|Gia dung|        3900|            1350.0|             2|
| Dien tu|       12500|1016.6666666666666|             3|
+--------+------------+------------------+--------------+



In [13]:
# 8. Sắp xếp (Giống df.sort_values())
print("Sắp xếp theo doanh thu giảm dần:")
summary_df.orderBy(F.col("TongDoanhThu").desc()).show()

Sắp xếp theo doanh thu giảm dần:
+--------+------------+------------------+--------------+
| DanhMuc|TongDoanhThu|      GiaTrungBinh|SoLuongMatHang|
+--------+------------+------------------+--------------+
| Dien tu|       12500|1016.6666666666666|             3|
|Gia dung|        3900|            1350.0|             2|
+--------+------------+------------------+--------------+



In [ ]:
# 9. Lưu kết quả ra file mới (Giống df.to_csv())
# Lưu ý: Spark sẽ tạo ra một thư mục chứa các file con vì nó xử lý phân tán
summary_df.write.mode("overwrite").csv("ket_qua_doanh_thu.csv", header=True)
print("Đã lưu file thành công!")

In [15]:
# Hiển thị giống pandas
df.limit(10).toPandas()

,TenSanpham,DanhMuc,Gia,SoLuong
0,Dien thoai,Dien tu,1000,2
1,Tu lanh,Gia dung,1500,1
2,Laptop,Dien tu,2000,5
3,May giat,Gia dung,1200,2
4,Chuot,Dien tu,50,10
